# 260512 Adaptive RAG 구현 1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w9_agent_rag/llm_260512_adaptive_rag_1.ipynb)

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-openai openai

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 강의 메모: Adaptive RAG 라우팅의 핵심

- **왜 나왔나**: 2024 논문. 그 전엔 모든 쿼리를 무지성으로 retriever에 던졌음. "1+1은?" 같은 질문도 벡터 스토어 뒤져 엉뚱한 청크를 끌어오니 **토큰·시간 낭비**. 쿼리 종류에 맞게 경로를 갈라주자는 발상.
- **랭그래프와의 관계**: 새 개념이 아니라 **랭그래프 응용**. 지난주 RAG 골격 사이사이에 "라우팅 노드"를 끼워 넣는 것. 같은 패턴으로 hallucination 체크·멀티홉 노드도 끼웠다 뺐다 가능.
- **논문 vs 요즘 구현**: 원논문은 T5 인코더를 **파인튜닝**해 분류기로 썼지만, 요즘 GPT-4o-mini는 파인튜닝 없이도 분류를 잘 함 → `with_structured_output(QueryAnalysis)`로 끝.
- **3-route 분기**: `direct`(LLM 자체 지식·1+1·데코레이터), `rag`(사내 문서·정책·매뉴얼), `web_search`(시세·뉴스·날씨). 실제 프로젝트에선 라우트가 훨씬 많아짐.
- **structured output이 사실상 전부**: 프롬프트로 "direct로 답해" 시키면 "direct입니다"/"처리입니다" 식으로 들쭉날쭉 → 파싱 불가. Pydantic `QueryAnalysis`(route/confidence/reasoning)로 **응답 포맷 강제**가 핵심.
- **멀티홉은 별도 트랙**: "내가 갔던 식당 셰프와 같이 일한 사람이 운영하는 식당 메뉴" 같이 hop 건너뛰는 질문은 벡터 유사도로 청크 따로따로 가져와선 못 풀음 → 마지막 주 **그래프 RAG**가 담당. Adaptive는 안 다룸.
- **confidence + 가중치**: LLM이 `direct`라 해도 confidence가 낮으면 못 믿음 → fallback. 거기에 `route_weight`(direct=1.0/rag=0.8/web=0.6)로 **호출 비용**까지 반영 → 비싼 경로는 더 확실할 때만 타게 만드는 장치.

## 강의 메모: 실무 팁

- **`@dataclass` 쓰는 이유**: `RoutingConfig`/`RoutingLog`처럼 데이터만 담는 클래스는 한 줄로 `__init__`/`__repr__`/타입힌트 자동 생성 → 비즈니스 로직에 집중. dict 디폴트는 `field(default_factory=lambda: {...})`로 안전하게.
- **ABC(추상 클래스) 패턴**: `RouteHandler(ABC)` + `@abstractmethod handle()`로 핸들러 인터페이스 통일. 라우트가 3→30→300개로 늘어도 **레지스트리 dict**에 한 줄 추가로 확장.
- **로그는 노가다 줄이는 핵심**: 라우팅은 헷갈리는 케이스가 많음. timestamp/predicted/actual/confidence/fallback_applied를 매번 누적 → threshold·프롬프트·라우트 정의를 **실데이터로 튜닝**. 안 찍으면 감으로 고침.
- **fallback 노드의 진짜 용도**: "그거 뭐야?" 같이 모호한 질문(confidence<0.5)엔 답하지 말고 **LLM에 구체 질문 3개 추천받아 사용자에게 되돌리기**. UX·토큰 양쪽 이득.
- **랭그래프 연결 흔한 실수**: 수강생 사례 — `fallback` 노드 자리에 실수로 `web_search_node`를 등록. 노드 추가·conditional_edges 매핑 dict·`add_edge(..., END)` 셋 다 키 이름 확인 필수.
- **이론 3줄, 구현 200줄**: "쿼리 분석 → 분기 → 실행"이 전부지만 랭그래프에 끼우면 State/노드/엣지/핸들러/엔진/로그로 코드가 부푼다. 각 노드는 결국 `dict in → dict out` 패턴 하나.